# 4. Physical stage from saved warmup
Load notebook 3's immutable warmup and exact inputs. Disable warmup and start physical factorization from those fields. Each outer round alternates ER updates with a shared charge/magnetization projection; set OUTER_ITERATIONS=0 for a **single final physical projection without detector updates**.

Saturation anchors are optional: list scan indices independently known to be saturated and their signs. Without two opposite saturated anchors the fitted magnetization is relative contrast, not an absolute calibrated magnetization. The material region defaults to the largest support component saved in notebook 3. Illumination and phase switches are independent.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
if HERE.name != "permanent_magnets":
    HERE = HERE / "permanent_magnets"
REPO = HERE.parent
sys.path.insert(0, str(REPO))
from permanent_magnets import workflow

from library import phase_retrieval_universal as universal

In [ ]:
PROCESSED = Path("/path/to/processed")  # Edit: same output folder in all four notebooks.
INPUT = PROCESSED / "03_warmup.h5"  # HDF5 input from the preceding notebook.
OUTPUT = PROCESSED / "04_physical.h5"  # HDF5 file written by this notebook; overwritten on rerun.

# Prepared hologram grid (must match the saved warmup)
CROP = 0  # Pixels removed from each edge; keep 0 in retrieval to preserve the saved grid.
BINNING = 1  # Detector binning factor; keep 1 for the prepared grid.
ROI = None  # Optional retrieval region; keep None to use the prepared grid.
HOLOGRAM_INTENSITY_CUTOFF_VMIN = -1  # -1 disables percentile background subtraction.

# Initialization: fields and modes are loaded from notebook 3

# Modes and coherence
PARTIAL_COHERENCE = False  # Enable partial-coherence updates; False uses coherent retrieval.
FFT_WORKERS = 1  # FFT threads used for each observation update.
COMMON_SECONDARY_MODES = True  # Share higher-mode objects across observations.

# Warmup: disabled because starting fields are loaded
WARMUP_NIT = 0  # 0 skips warmup and starts from the saved fields.

# Physical model and alternating steps
PROJECTION_MODEL = "physical_factorized"  # Physical factorization model; "none" disables physical fitting.
OUTER_ITERATIONS = 10        # 0: projection only; >0: ER + physical projection rounds.
INNER_MODE = "ER"  # Detector/support update algorithm between physical projections.
INNER_ITERATIONS = 50  # Retrieval iterations per observation in each outer round.
BETA_MODE = "const"  # Schedule controlling the feedback parameter beta.
BETA_ZERO = .5  # Initial feedback parameter beta.
ALPHA_ZERO = .5  # Initial TV regularization step size.
TV_FREQ = 10  # Apply TV regularization every this many retrieval iterations.
AVERAGE_IMG = 1  # Number of final iteration fields averaged per update.
PHYSICAL_ITERATIONS = 50  # Alternating charge/magnetization fit iterations per projection.
PROJECTION_RELAXATION = 1.  # Physical projection mixing; 1 fully applies the fitted model.
FINAL_PROJECTION_RELAXATION = 1.  # Mixing for the final physical projection; 0 skips it.
FINAL_FOURIER_CONSTRAINT = False  # Reapply measured amplitudes after the physical fit if True.

# Material and magnetization constraints
SATURATED_SCAN_INDICES = []  # Example: [1, 2], only when independently known to saturate.
SATURATION_SIGNS = []        # Corresponding +1 or -1 values.
FIT_MATERIAL_THICKNESS = False  # Fit thickness if True; otherwise retain the saved material map.
PHYSICAL_PROJECTION_OBJECT_ROI = True  # Fit on the material region and shift-reference pixels.
PHYSICAL_PHASE_REFERENCE = True  # Choose consistent logarithm phase branches.
INITIAL_BINARY_MAGNETIZATION = False  # Continuous measured-contrast initializer by default.
BINARY_MAGNETIZATION = False  # Constrain every fitted magnetization update to discrete values.
CLIP_MAGNETIZATION = False  # Bound fitted magnetization to [-1, 1] if True.
MAGNETIZATION_TREND = "none"  # none, increasing, decreasing; observation order.


# Illumination fits and spatial weighting
ALIGN_GLOBAL_PHASE = True  # Align observation phase gauges before physical fitting.
FIT_ILLUMINATION_INTENSITY = True  # Fit relative illumination using material-free support pixels.
FIT_APERTURE_ILLUMINATION = True  # Fit separate illumination factors for reference holes.
OBJECT_ILLUMINATION_MODE = "linear_plane"  # RH-based gain extrapolation to object.
APERTURE_FACTOR_TYPE = "complex"  # "complex" fits amplitude and phase; "amplitude" fits amplitude only.
PHYSICAL_SPATIAL_WEIGHTING = "exit_wave_intensity"  # None disables spatial weighting.


In [ ]:
with h5py.File(INPUT, "r") as h:
    warmup = h["warmup_fields"][()]
    holograms = h["holograms"][()]
    mask_pixel = h["mask_pixel"][()].astype(bool)
    support = h["supportmask"][()]
    material = h["material_thickness"][()]
    modes = h["modes"][()].tolist()
    scan_indices = h["scan_indices"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
    state_labels = h["state_labels"].asstr()[()].tolist()
    assert h.attrs["crop"] == 0 and h.attrs["binning"] == 1
assert len(SATURATED_SCAN_INDICES) == len(SATURATION_SIGNS)
assert len(set(SATURATED_SCAN_INDICES)) == len(SATURATED_SCAN_INDICES)
assert set(SATURATED_SCAN_INDICES).issubset(set(scan_indices))
assert all(sign in (-1, 1) for sign in SATURATION_SIGNS)
saturated_states = {f"scan_{index}": float(sign) for index, sign in zip(SATURATED_SCAN_INDICES, SATURATION_SIGNS)}
calibrated = set(SATURATION_SIGNS) == {-1, 1}
print("Calibration:", "two saturated endpoints" if calibrated else "relative contrast (no two opposite anchors)")
reference_index = list(scan_indices).index(SATURATED_SCAN_INDICES[0]) if SATURATED_SCAN_INDICES else 0

recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    crop=CROP,
    binning=BINNING,
    roi=ROI,
    hologram_intensity_cutoff_vmin=HOLOGRAM_INTENSITY_CUTOFF_VMIN,
    partial_coherence=PARTIAL_COHERENCE,
    constrain_nonphysical_modes_common=COMMON_SECONDARY_MODES,
    warmup_Nit=WARMUP_NIT,
    projection_model=PROJECTION_MODEL,
    outer_iterations=OUTER_ITERATIONS,
    inner_mode=INNER_MODE,
    inner_Nit=INNER_ITERATIONS,
    beta_mode=BETA_MODE,
    beta_zero=BETA_ZERO,
    alpha_zero=ALPHA_ZERO,
    TV_freq=TV_FREQ,
    average_img=AVERAGE_IMG,
    physical_iterations=PHYSICAL_ITERATIONS,
    projection_relaxation=PROJECTION_RELAXATION,
    final_projection_relaxation=FINAL_PROJECTION_RELAXATION,
    final_fourier_constraint=FINAL_FOURIER_CONSTRAINT,
    fit_material_thickness=FIT_MATERIAL_THICKNESS,
    physical_projection_object_roi=PHYSICAL_PROJECTION_OBJECT_ROI,
    physical_phase_reference=PHYSICAL_PHASE_REFERENCE,
    initial_binary_magnetization=INITIAL_BINARY_MAGNETIZATION,
    binary_magnetization=BINARY_MAGNETIZATION,
    clip_magnetization=CLIP_MAGNETIZATION,
    magnetization_trend=MAGNETIZATION_TREND,
    align_global_phase=ALIGN_GLOBAL_PHASE,
    fit_illumination_intensity=FIT_ILLUMINATION_INTENSITY,
    fit_aperture_illumination=FIT_APERTURE_ILLUMINATION,
    object_illumination_mode=OBJECT_ILLUMINATION_MODE,
    aperture_factor_type=APERTURE_FACTOR_TYPE,
    physical_spatial_weighting=PHYSICAL_SPATIAL_WEIGHTING,
    fft_workers=FFT_WORKERS,
    modes=modes,
    material_thickness=material,
    saturated_states=saturated_states,
    illumination_intensity_reference_observation=reference_index,
    projection_every=len(state_labels),
)
fields, received_start, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, support, state_labels=state_labels,
    energy_labels=energies, polarization_coefficients=polarizations,
    illumination_labels=["beam0"]*len(state_labels), universal_recipe=recipe,
    start_fields=warmup.copy(), phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel,
)
np.testing.assert_allclose(received_start, warmup, rtol=1e-14, atol=0.)
assert list(components["state_names"]) == state_labels
print("Physical model log-fit RMS:", components["fit_residual_rms"])

In [ ]:
objects, intensity_residual = workflow.field_diagnostics(fields, holograms, bsmasks, state_labels)
# Library component maps use the shifted fitting frame; center once for display.
magnetization = np.fft.fftshift(components["magnetization"], axes=(-2, -1))
active = material > 0
mean_magnetization = magnetization[:, active].mean(axis=1)
limit = max(1., float(np.max(abs(magnetization[:, active]))))
workflow.plot_stack(np.where(active, magnetization, np.nan), state_labels, "Reduced magnetization" if calibrated else "Relative magnetic contrast", "RdBu_r", -limit, limit)
common_wave = np.fft.fftshift(components["common_exit_waves"], axes=(-2, -1))
workflow.plot_stack(abs(common_wave), ["beam0"], "Shared illumination/reference wave amplitude")
workflow.plot_stack(np.angle(common_wave), ["beam0"], "Shared illumination/reference wave phase", "twilight", -np.pi, np.pi)
print("Charge response:", components["charge_response"])
print("Magnetic response:", components["magnetic_response"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(mean_magnetization, "o-"); axes[0].set(xlabel="Observation (acquisition order)", ylabel="Mean magnetization" if calibrated else "Mean relative contrast")
illumination = np.asarray(components.get("illumination_intensity_factors", np.ones(len(state_labels))))
axes[1].plot(illumination, "o-"); axes[1].set(xlabel="Observation", ylabel="Relative illumination intensity")
plt.show()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    h.attrs["source_file"] = str(INPUT.resolve())
    h.attrs["calibration"] = "two_saturated_endpoints" if calibrated else "relative_contrast"
    h.attrs["component_coordinates"] = "components: solver fitting frame; magnetization: centered display frame"
    for name, array in [("fields", fields), ("warmup_fields", warmup), ("objects", objects),
                        ("holograms", holograms), ("mask_pixel", mask_pixel), ("bsmasks", bsmasks),
                        ("supportmask", support), ("material_thickness", material), ("modes", modes),
                        ("scan_indices", scan_indices), ("energy_eV", energies), ("polarization", polarizations),
                        ("magnetization", magnetization), ("mean_magnetization", mean_magnetization),
                        ("intensity_residual", intensity_residual), ("saturated_scan_indices", SATURATED_SCAN_INDICES),
                        ("saturation_signs", SATURATION_SIGNS)]:
        h.create_dataset(name, data=array, compression="gzip")
    h.create_dataset("state_labels", data=state_labels, dtype=h5py.string_dtype())
    workflow.save_tree(h.create_group("components"), components)
    workflow.save_tree(h.create_group("errors"), errors)
    # Recipe settings and material are saved for reproducibility, not used as a UI.
    workflow.save_tree(h.create_group("recipe"), recipe)
print("Saved", OUTPUT)